# Self-Play V9: Openings (2200 sims, 750 turns)

Main data push: elite opening decisions at deep search.
750 turns at 2200 sims per game. Short games = many seeds = diversity.

**Upload to Drive (`MyDrive/alphatrain/`):**
1. `colorlines_selfplay_train.tar.gz` — code
2. `pillar2w2_epoch_10.pt` — model

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/alphatrain'

!cp {DRIVE}/colorlines_selfplay_train.tar.gz /content/
!cd /content && tar xzf colorlines_selfplay_train.tar.gz

MODEL_NAME = 'pillar2w2_epoch_10.pt'

os.makedirs('/content/alphatrain/data', exist_ok=True)
!cp {DRIVE}/{MODEL_NAME} /content/alphatrain/data/model.pt
print(f'Model: {os.path.getsize("/content/alphatrain/data/model.pt")/1e6:.0f} MB')

!pip install -q numpy numba scipy

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

from alphatrain.evaluate import load_model
net, ms = load_model('/content/alphatrain/data/model.pt', 'cpu')
del net
print(f'Model OK, max_score={ms}')

In [ ]:
# === OPENINGS BATCH 1 ===
# 2200 sims, 750-turn cap — elite early-game decisions
# Instance 1: 500000-505000
# Instance 2: 505000-510000
SEED_START = 500000
SEED_END = 505000
# ========================

SIMS = 2200
WORKERS = 24
BATCH_SIZE = 128
MAX_TURNS = 750
SAVE_DIR = f'{DRIVE}/selfplay_v9_openings'

os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Seeds: {SEED_START}-{SEED_END-1} ({SEED_END-SEED_START} games)')
print(f'Sims: {SIMS}, Cap: {MAX_TURNS} turns')
print(f'Expected: ~{(SEED_END-SEED_START)*MAX_TURNS/1e6:.1f}M states')

!python -m alphatrain.scripts.selfplay \
    --model /content/alphatrain/data/model.pt \
    --seed-start {SEED_START} --seed-end {SEED_END} \
    --sims {SIMS} --batch-size {BATCH_SIZE} \
    --save-dir {SAVE_DIR} \
    --workers {WORKERS} --device cuda \
    --temperature-moves 5 \
    --max-turns {MAX_TURNS}